# Train a dedicated YOLO26s earphone detector on Kaggle

This notebook builds a one-class `earphone` detector from the Datazeft `Earphone` and `headset` annotations. It accepts both YOLO bbox rows and polygon rows, balances positive images with hard negatives, trains at high resolution, and exports `yolo26s_earphone_best.pt`.

Keep this checkpoint separate from the paper segmentation checkpoint and `best (1).pt`.

In [ ]:
%pip install -q -U "ultralytics>=8.4.0,<9" pyyaml pandas matplotlib


In [ ]:
from __future__ import annotations
import hashlib, json, math, os, random, shutil
from collections import Counter, defaultdict
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import torch, yaml
from IPython.display import FileLink, display
from PIL import Image, ImageDraw
from ultralytics import YOLO, __version__ as ultralytics_version

SEED = 42
random.seed(SEED)
print('Ultralytics:', ultralytics_version)
print('PyTorch:', torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU accelerator in Kaggle Notebook options.')
print('GPU:', torch.cuda.get_device_name(0))


## 1. Locate Datazeft and optional custom earphone data

In [ ]:
KAGGLE_INPUT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working/earphone_yolo26s')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def dataset_roots():
    roots = []
    for yaml_path in KAGGLE_INPUT.rglob('data.yaml'):
        root = yaml_path.parent
        if (root / 'train' / 'images').is_dir():
            roots.append(root)
    return sorted(set(roots), key=str)

ROOTS = dataset_roots()
for index, root in enumerate(ROOTS):
    print(index, root)
DATAZEFT_ROOT = next((root for root in ROOTS if 'datazeft' in str(root).lower()), None)
if DATAZEFT_ROOT is None:
    raise FileNotFoundError('Attach ahmedezzat02/datazeft with Add Input.')
CUSTOM_EARPHONE_ROOT = next((root for root in ROOTS if root != DATAZEFT_ROOT and any(token in str(root).lower() for token in ('ear', 'headset', 'earbud'))), None)
print('Datazeft:', DATAZEFT_ROOT)
print('Optional custom earphone dataset:', CUSTOM_EARPHONE_ROOT)


## 2. Convert to a balanced one-class detection dataset

Polygon annotations are converted to their enclosing detection boxes. Images without earphones are sampled as hard negatives instead of overwhelming the positives.

In [ ]:
OUTPUT_DATASET = WORK_ROOT / 'dataset'
if OUTPUT_DATASET.exists():
    shutil.rmtree(OUTPUT_DATASET)
SPLIT_FOLDERS = {'train': 'train', 'val': 'valid', 'test': 'test'}
for split in SPLIT_FOLDERS:
    (OUTPUT_DATASET / split / 'images').mkdir(parents=True, exist_ok=True)
    (OUTPUT_DATASET / split / 'labels').mkdir(parents=True, exist_ok=True)

def yaml_names(root):
    payload = yaml.safe_load((root / 'data.yaml').read_text(encoding='utf-8'))
    raw = payload.get('names', {})
    if isinstance(raw, list):
        return {index: str(name) for index, name in enumerate(raw)}
    return {int(index): str(name) for index, name in raw.items()}

def normalize(value):
    return value.lower().strip().replace('-', '_').replace(' ', '_')

EARPHONE_ALIASES = {'earphone', 'earphones', 'earbud', 'earbuds', 'headphone', 'headphones', 'headset'}
stats = Counter()
groups_by_split = defaultdict(set)

def parse_target_rows(label_path, target_ids):
    rows = []
    if not label_path.is_file():
        return rows
    for raw_line in label_path.read_text(encoding='utf-8').splitlines():
        parts = raw_line.split()
        if len(parts) < 5:
            stats['invalid_rows'] += 1
            continue
        try:
            class_id = int(float(parts[0]))
            coordinates = [float(value) for value in parts[1:]]
        except ValueError:
            stats['invalid_rows'] += 1
            continue
        if class_id not in target_ids:
            continue
        if not all(math.isfinite(value) and 0 <= value <= 1 for value in coordinates):
            stats['invalid_rows'] += 1
            continue
        if len(coordinates) == 4:
            x, y, width, height = coordinates
            stats['bbox_rows'] += 1
        elif len(coordinates) >= 6 and len(coordinates) % 2 == 0:
            xs, ys = coordinates[0::2], coordinates[1::2]
            x_min, x_max, y_min, y_max = min(xs), max(xs), min(ys), max(ys)
            x, y = (x_min + x_max) / 2, (y_min + y_max) / 2
            width, height = x_max - x_min, y_max - y_min
            stats['polygon_rows_converted'] += 1
        else:
            stats['invalid_rows'] += 1
            continue
        if width <= 0 or height <= 0:
            stats['invalid_rows'] += 1
            continue
        rows.append(f'0 {x:.8f} {y:.8f} {width:.8f} {height:.8f}')
    return rows

def collect_source(root, source_tag):
    names = yaml_names(root)
    target_ids = {index for index, name in names.items() if normalize(name) in EARPHONE_ALIASES}
    if not target_ids:
        raise RuntimeError(f'No earphone/headset class in {root}: {names}')
    collected = {}
    for split, folder in SPLIT_FOLDERS.items():
        positives, negatives = [], []
        image_dir, label_dir = root / folder / 'images', root / folder / 'labels'
        for image_path in sorted(image_dir.rglob('*')):
            if image_path.suffix.lower() not in IMAGE_SUFFIXES:
                continue
            rows = parse_target_rows(label_dir / f'{image_path.stem}.txt', target_ids)
            item = (image_path, rows)
            (positives if rows else negatives).append(item)
        collected[split] = (positives, negatives)
        print(source_tag, split, 'positive images=', len(positives), 'negative candidates=', len(negatives))
    return collected

def materialize(collected, source_tag, negative_ratio):
    rng = random.Random(SEED + sum(ord(character) for character in source_tag))
    for split, (positives, negatives) in collected.items():
        selected_negatives = rng.sample(negatives, min(len(negatives), round(len(positives) * negative_ratio)))
        for image_path, rows in positives + selected_negatives:
            digest = hashlib.sha1(str(image_path).encode('utf-8')).hexdigest()[:10]
            target_stem = f'{source_tag}_{digest}_{image_path.stem}'
            target_image = OUTPUT_DATASET / split / 'images' / f'{target_stem}{image_path.suffix.lower()}'
            target_label = OUTPUT_DATASET / split / 'labels' / f'{target_stem}.txt'
            try:
                os.symlink(image_path, target_image)
            except OSError:
                shutil.copy2(image_path, target_image)
            target_label.write_text(('\n'.join(rows) + '\n') if rows else '', encoding='utf-8')
            stats[f'{split}_images'] += 1
            stats[f'{split}_positive_images'] += bool(rows)
            stats[f'{split}_instances'] += len(rows)
            group = image_path.name.split('.rf.', 1)[0]
            groups_by_split[f'{source_tag}:{group}'].add(split)

materialize(collect_source(DATAZEFT_ROOT, 'public'), 'public', negative_ratio=1.5)
if CUSTOM_EARPHONE_ROOT is not None:
    materialize(collect_source(CUSTOM_EARPHONE_ROOT, 'custom'), 'custom', negative_ratio=2.0)

leaked = {group: sorted(splits) for group, splits in groups_by_split.items() if len(splits) > 1}
display(pd.Series(stats).sort_index().to_frame('count'))
print('Cross-split source groups:', len(leaked))
if leaked:
    print('WARNING: augmented variants cross splits; metrics may be optimistic.')
    print(list(leaked.items())[:10])
for split in SPLIT_FOLDERS:
    assert stats[f'{split}_positive_images'] > 0, f'No positive images in {split}'
assert stats['invalid_rows'] == 0, 'Fix invalid annotation rows before training.'

DATA_YAML = WORK_ROOT / 'earphone.yaml'
DATA_YAML.write_text(yaml.safe_dump({'path': str(OUTPUT_DATASET), 'train': 'train/images', 'val': 'val/images', 'test': 'test/images', 'names': {0: 'earphone'}}, sort_keys=False), encoding='utf-8')
print(DATA_YAML.read_text(encoding='utf-8'))


## 3. Audit random labels before training

In [ ]:
def list_images(directory):
    return sorted(path for path in directory.iterdir() if path.suffix.lower() in IMAGE_SUFFIXES)

def draw_boxes(image_path):
    image = Image.open(image_path).convert('RGB')
    draw, (width, height) = ImageDraw.Draw(image), image.size
    label_path = image_path.parent.parent / 'labels' / f'{image_path.stem}.txt'
    for raw_line in label_path.read_text(encoding='utf-8').splitlines():
        _, x, y, box_width, box_height = map(float, raw_line.split())
        x1, y1 = (x - box_width / 2) * width, (y - box_height / 2) * height
        x2, y2 = (x + box_width / 2) * width, (y + box_height / 2) * height
        draw.rectangle((x1, y1, x2, y2), outline='red', width=3)
        draw.text((x1, max(0, y1 - 14)), 'earphone', fill='red')
    return image

positive_train = [path for path in list_images(OUTPUT_DATASET / 'train' / 'images') if (OUTPUT_DATASET / 'train' / 'labels' / f'{path.stem}.txt').stat().st_size > 0]
samples = random.sample(positive_train, min(12, len(positive_train)))
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for axis in axes.flat:
    axis.axis('off')
for axis, image_path in zip(axes.flat, samples):
    axis.imshow(draw_boxes(image_path))
    axis.set_title(image_path.name[:45], fontsize=8)
plt.tight_layout()


## 4. Two-stage fine-tuning at 960 pixels

Stage 1 trains the new one-class head while freezing the early backbone. Stage 2 unfreezes everything with a lower learning rate. Auto-batch uses approximately 60% of available GPU memory.

In [ ]:
SMOKE_TEST = False
STAGE1_EPOCHS = 2 if SMOKE_TEST else 15
STAGE2_EPOCHS = 2 if SMOKE_TEST else 85
FRACTION = 0.04 if SMOKE_TEST else 1.0

base_model = YOLO('yolo26s.pt')
base_model.train(
    data=str(DATA_YAML), epochs=STAGE1_EPOCHS, patience=10, imgsz=960, batch=-1,
    device=0, workers=4, project=str(WORK_ROOT), name='earphone_stage1', exist_ok=True,
    pretrained=True, freeze=10, optimizer='auto', amp=True, seed=SEED,
    deterministic=True, cache=False, fraction=FRACTION, plots=True,
    hsv_h=0.015, hsv_s=0.50, hsv_v=0.35, degrees=8.0, translate=0.10,
    scale=0.40, shear=2.0, perspective=0.0005, fliplr=0.50, mosaic=0.60,
    mixup=0.05, close_mosaic=5, multi_scale=0.20,
)
STAGE1_BEST = WORK_ROOT / 'earphone_stage1' / 'weights' / 'best.pt'
assert STAGE1_BEST.is_file()

fine_model = YOLO(str(STAGE1_BEST))
fine_model.train(
    data=str(DATA_YAML), epochs=STAGE2_EPOCHS, patience=20, imgsz=960, batch=-1,
    device=0, workers=4, project=str(WORK_ROOT), name='earphone_stage2', exist_ok=True,
    pretrained=True, optimizer='auto', lr0=0.001, cos_lr=True, amp=True,
    seed=SEED, deterministic=True, cache=False, fraction=FRACTION, plots=True,
    save=True, save_period=10, hsv_h=0.015, hsv_s=0.45, hsv_v=0.30,
    degrees=6.0, translate=0.08, scale=0.30, shear=1.5, perspective=0.0003,
    fliplr=0.50, mosaic=0.40, mixup=0.0, close_mosaic=10, multi_scale=0.15,
)
RUN_DIR = WORK_ROOT / 'earphone_stage2'
BEST_PT = RUN_DIR / 'weights' / 'best.pt'
LAST_PT = RUN_DIR / 'weights' / 'last.pt'
assert BEST_PT.is_file()
print('Best checkpoint:', BEST_PT)
# Resume in the same Kaggle session with: YOLO(str(LAST_PT)).train(resume=True)


## 5. Evaluate recall and preview hard examples

In [ ]:
best_model = YOLO(str(BEST_PT))
metrics_e2e = best_model.val(data=str(DATA_YAML), split='test', imgsz=960, batch=8, device=0, conf=0.001, iou=0.60, plots=True, project=str(WORK_ROOT), name='earphone_test_e2e', end2end=True)
metrics_recall = best_model.val(data=str(DATA_YAML), split='test', imgsz=960, batch=8, device=0, conf=0.001, iou=0.60, plots=True, project=str(WORK_ROOT), name='earphone_test_recall', end2end=False)
summary = pd.DataFrame([
    {'head': 'end-to-end', 'mAP50': metrics_e2e.box.map50, 'mAP50-95': metrics_e2e.box.map, 'precision': metrics_e2e.box.mp, 'recall': metrics_e2e.box.mr},
    {'head': 'one-to-many', 'mAP50': metrics_recall.box.map50, 'mAP50-95': metrics_recall.box.map, 'precision': metrics_recall.box.mp, 'recall': metrics_recall.box.mr},
])
display(summary)

test_images = list_images(OUTPUT_DATASET / 'test' / 'images')
preview_sources = random.sample(test_images, min(16, len(test_images)))
predictions = best_model.predict(source=[str(path) for path in preview_sources], imgsz=1280, conf=0.10, iou=0.55, max_det=50, device=0, end2end=False, verbose=False)
fig, axes = plt.subplots(4, 4, figsize=(18, 18))
for axis in axes.flat:
    axis.axis('off')
for axis, result in zip(axes.flat, predictions):
    axis.imshow(result.plot()[:, :, ::-1])
    axis.set_title(Path(result.path).name[:40], fontsize=8)
plt.tight_layout()


## 6. Package the checkpoint

Start production tuning around confidence 0.15-0.25 with temporal confirmation; select the final threshold using target-camera validation, not training images.

In [ ]:
FINAL_PT = Path('/kaggle/working/yolo26s_earphone_best.pt')
shutil.copy2(BEST_PT, FINAL_PT)
metadata = {
    'task': 'detection', 'base_model': 'yolo26s.pt', 'class_names': ['earphone'],
    'train_image_size': 960, 'recommended_roi_inference_size': 1280,
    'recommended_end2end': False, 'recommended_confidence_range': [0.15, 0.25],
    'test_one_to_many': {'map50': float(metrics_recall.box.map50), 'map50_95': float(metrics_recall.box.map), 'precision': float(metrics_recall.box.mp), 'recall': float(metrics_recall.box.mr)},
}
METADATA_JSON = Path('/kaggle/working/yolo26s_earphone_metadata.json')
METADATA_JSON.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
ARCHIVE_ZIP = Path(shutil.make_archive('/kaggle/working/yolo26s_earphone_training_run', 'zip', root_dir=RUN_DIR))
display(FileLink(str(FINAL_PT)))
display(FileLink(str(METADATA_JSON)))
display(FileLink(str(ARCHIVE_ZIP)))
